# 6. Respuesta a las Preguntas de Negocio

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = "../data"
CSV_PATH = f"{DATA_DIR}/GBvideos_cc50_202101.csv"
JSON_PATH = f"{DATA_DIR}/GB_category_id.json"
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# Carga, mapeo de categorías y variables derivadas necesarias para el negocio
df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    categories_data = json.load(f)
category_map = {
    int(item["id"]): item["snippet"]["title"] for item in categories_data["items"]
}
df["category_name"] = df["category_id"].map(category_map).fillna("Unknown")
df["trending_date"] = pd.to_datetime(df["trending_date"], errors="coerce")

df["like_dislike_ratio"] = df["likes"] / (df["dislikes"] + 1)
df["comments_per_view"] = df["comment_count"] / (df["views"] + 1)
df["engagement_rate"] = (df["likes"] + df["dislikes"] + df["comment_count"]) / (df["views"] + 1)
print("Shape:", df.shape)

## P1. ¿Qué categorías aparecen más en tendencias?

In [ ]:
# P1. Categorías más frecuentes en tendencias
top_cats = df["category_name"].value_counts().head(10)
print(top_cats)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, palette="viridis",
            hue=top_cats.index, legend=False)
plt.title("P1 · Top 10 categorías en tendencias")
plt.xlabel("Frecuencia")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/business_p1_categories.png", dpi=300, bbox_inches="tight")
plt.show()

## P2. ¿Qué categorías tienen más likes?

In [ ]:
likes_by_cat = df.groupby('category_name')['likes'].agg(['mean','median','count']).sort_values('mean', ascending=False).head(10)
display(likes_by_cat)

## P3. Ranking de categorías por ratio likes/dislikes

In [ ]:
ratio_by_cat = df.groupby('category_name')['like_dislike_ratio'].agg(['mean','median']).sort_values('mean', ascending=False).head(10)
display(ratio_by_cat)

## P4. Ranking de categorías por ratio views/comments

In [ ]:
inv_ratio = df.groupby('category_name')['comments_per_view'].agg(['mean','median']).sort_values('mean', ascending=False).head(10)
display(inv_ratio)

## P5. Tendencia temporal

In [ ]:
# P5. Tendencia temporal de vistas promedio por semana
serie = df.groupby(df["trending_date"].dt.to_period("W"))["views"].mean().reset_index()
serie["trending_date"] = serie["trending_date"].dt.to_timestamp()
serie["media_movil"] = serie["views"].rolling(4, min_periods=1).mean()

plt.figure(figsize=(12, 5))
sns.lineplot(data=serie, x="trending_date", y="views", label="Vistas prom.", color="navy")
sns.lineplot(data=serie, x="trending_date", y="media_movil", label="Media móvil (4)", color="red")
plt.title("P5 · Evolución temporal de vistas")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/business_p5_trend.png", dpi=300, bbox_inches="tight")
plt.show()

## P6. Canales más frecuentes

In [ ]:
top_channels = df['channel_title'].value_counts().head(20)
print(top_channels)

## P7. Mapa geográfico

In [ ]:
# P7. Distribución geográfica por región (state)
if {"lat", "lon", "state"}.issubset(df.columns):
    geo = df.groupby("state").agg(
        videos=("video_id", "count"),
        views_prom=("views", "mean"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
    ).sort_values("videos", ascending=False)
    display(geo.head(15))

    # Mapa interactivo opcional con folium (heatmap de vistas)
    try:
        import folium
        from folium.plugins import HeatMap
        m = folium.Map(location=[54, -2], zoom_start=5)
        HeatMap(df[["lat", "lon", "views"]].dropna().values.tolist(), radius=10).add_to(m)
        m.save(f"{FIG_DIR}/business_p7_heatmap.html")
        print("Mapa guardado en", f"{FIG_DIR}/business_p7_heatmap.html")
    except ImportError:
        print("folium no está instalado; se muestra solo la tabla por región.")
else:
    print("No hay columnas geográficas disponibles.")

## P8. Sentimiento de títulos (proxy de comentarios)

In [ ]:
# P8. Sentimiento de los títulos como proxy de percepción (NLP básico con VADER)
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()
    df["sentiment_compound"] = df["title"].astype(str).apply(
        lambda t: analyzer.polarity_scores(t)["compound"]
    )
except ImportError:
    # Alternativa con TextBlob si VADER no está disponible
    from textblob import TextBlob
    df["sentiment_compound"] = df["title"].astype(str).apply(
        lambda t: TextBlob(t).sentiment.polarity
    )

df["sentiment_label"] = pd.cut(
    df["sentiment_compound"],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=["negative", "neutral", "positive"],
)
print(df["sentiment_label"].value_counts(normalize=True).round(3))

In [ ]:
# Relación entre sentimiento y vistas
df.groupby('sentiment_label')['views'].mean()

## P9. Resumen del modelo predictivo

In [ ]:
# P9. El modelo predictivo de vistas se desarrolla en 07_modeling.ipynb.
MODEL_CSV = f"{DATA_DIR}/processed/gb_videos_model.csv"
if os.path.exists(MODEL_CSV):
    model_df = pd.read_csv(MODEL_CSV)
    print("Dataset de modelado listo:", model_df.shape)
else:
    print("Ejecuta 05_feature_engineering.ipynb para generar el dataset de modelado.")